# 全局联邦学习（GFL）结果分析

用于浏览全部 GFL 结果、展示单次运行、比较多个算法，以及比较单算法的多组参数。通用读取和绘图逻辑位于 `utils/results.py`。

In [ ]:
import sys
from collections import Counter
from pathlib import Path

current = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in (current, *current.parents) if (path / 'pyproject.toml').is_file())
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from utils.results import (
    algorithm_names, configure_plots, discover_runs, load_checkpoint,
    plot_algorithm_comparison, plot_algorithm_runs,
    plot_parameter_comparison, plot_run, read_log, runs_table,
    select_runs, show_run_details,
)

configure_plots()
GFL_ALGORITHMS = algorithm_names(PROJECT_ROOT / 'algo' / 'gfl')
GFL_RUNS = discover_runs(PROJECT_ROOT, algorithms=GFL_ALGORITHMS)
print(f'GFL 运行 {len(GFL_RUNS)} 次。')
print(dict(sorted(Counter(run.algorithm for run in GFL_RUNS).items())))

## 1. 结果概览与筛选

筛选键可以是 `algorithm/status/run_id/trial`，也可以是 `args.json` 中的任意参数。集合表示多选，可调用对象可表达范围条件。

In [ ]:
runs_table(GFL_RUNS, limit=30)

# 示例：
# selected = select_runs(GFL_RUNS, algorithm='fedavg', dataset='cifar10', status='completed')
# runs_table(selected, limit=None)

## 2. 单次运行或单个算法展示

In [ ]:
# selected = select_runs(GFL_RUNS, algorithm='fedavg', status='completed')
# show_run_details(selected[0], GFL_RUNS)
# plot_run(selected[0], GFL_RUNS)  # 自动绘制全部数值指标和累计耗时
# plot_algorithm_runs(GFL_RUNS, 'fedavg', filters={'dataset': 'cifar10'})

## 3. 多算法对比

同一算法存在多个 trial 时绘制均值与标准差阴影。请用 `filters` 固定实验控制变量。

In [ ]:
# fig, axis, compared = plot_algorithm_comparison(
#     GFL_RUNS, ['fedavg', 'fedprox', 'feddyn'],
#     metric='accuracy',
#     filters={'dataset': 'cifar10', 'partition': 'dirichlet', 'dir': 0.1, 'status': 'completed'},
# )

## 4. 单算法参数对比

`vary` 可传单个参数名或参数名列表。函数会提示除对比参数之外仍有差异的配置。

In [ ]:
# fig, axis, groups = plot_parameter_comparison(
#     GFL_RUNS, 'fedfm', ['anchor_loss', 'alpha_coef'],
#     filters={'dataset': 'cifar10', 'status': 'completed'},
# )

## 5. 日志与 checkpoint

日志和 checkpoint 均按需读取；checkpoint 固定映射到 CPU。

In [ ]:
# run = GFL_RUNS[0]
# print(read_log(run, GFL_RUNS, tail=20))
# checkpoint = load_checkpoint(run, GFL_RUNS)